# MouseTrap — YOLOv8n Training for Camera-Trap Rodent Detection

Trains a YOLOv8n (nano) model on real camera-trap rodent datasets for server-side classification.

**Classes:** `mouse`, `rat`, `cat`, `dog`, `person`, `bird`, `empty`

**Datasets:**
- Channel Islands Camera Traps (LILA) — 82K images with COCO bbox annotations
- Roboflow rodent datasets — pre-annotated in YOLO format
- COCO 2017 — person, cat, dog, bird annotations

**Runtime:** ~30-60 min on Google Colab free tier (T4 GPU)

**Output:** `best.pt` — YOLOv8n fine-tuned weights

In [ ]:
# Cell 1: Setup — Install dependencies
!pip install -q ultralytics fiftyone pycocotools gdown

import os
import json
import shutil
import random
from pathlib import Path
from collections import defaultdict

# Working directory
WORK_DIR = Path('/content/mousetrap_yolo')
WORK_DIR.mkdir(exist_ok=True)
DATASET_DIR = WORK_DIR / 'dataset'
DATASET_DIR.mkdir(exist_ok=True)

# Class mapping for our model
CLASS_NAMES = ['mouse', 'rat', 'cat', 'dog', 'person', 'bird']
CLASS_TO_ID = {name: i for i, name in enumerate(CLASS_NAMES)}

print(f'Classes: {CLASS_NAMES}')
print(f'Working dir: {WORK_DIR}')

In [ ]:
# Cell 2: Download Channel Islands Camera Traps dataset
# This dataset has ~82K images with COCO-format bounding box annotations
# containing rodent species from the Channel Islands.

import urllib.request
import zipfile

CI_DIR = WORK_DIR / 'channel_islands'
CI_DIR.mkdir(exist_ok=True)

# COCO annotations for Channel Islands Camera Traps
CI_ANNOTATIONS_URL = 'https://lilablobssc.blob.core.windows.net/channel-islands-camera-traps/channel-islands-camera-traps-1.0.coco.json.zip'

ann_zip = CI_DIR / 'annotations.zip'
if not ann_zip.exists():
    print('Downloading Channel Islands annotations...')
    urllib.request.urlretrieve(CI_ANNOTATIONS_URL, ann_zip)

ann_json = CI_DIR / 'channel-islands-camera-traps-1.0.coco.json'
if not ann_json.exists():
    print('Extracting annotations...')
    with zipfile.ZipFile(ann_zip) as zf:
        zf.extractall(CI_DIR)

# Load and inspect annotations
print('Loading COCO annotations...')
with open(ann_json) as f:
    ci_coco = json.load(f)

# Map category IDs to names
ci_categories = {c['id']: c['name'] for c in ci_coco['categories']}
print(f'\nChannel Islands categories ({len(ci_categories)}):')
for cid, name in sorted(ci_categories.items()):
    print(f'  {cid}: {name}')

print(f'\nTotal images: {len(ci_coco["images"])}')
print(f'Total annotations: {len(ci_coco["annotations"])}')

In [ ]:
# Cell 3: Map Channel Islands categories to our classes and download images
# Channel Islands has species like 'deer mouse', 'island fox', etc.
# We map rodent species to 'mouse' or 'rat'.

# Mapping from Channel Islands category names to our classes
CI_CLASS_MAP = {}
RODENT_KEYWORDS = ['mouse', 'rat', 'vole', 'shrew', 'rodent', 'peromyscus']
CAT_KEYWORDS = ['cat', 'feline']
DOG_KEYWORDS = ['dog', 'fox', 'canine']  # island foxes -> dog category
BIRD_KEYWORDS = ['bird', 'jay', 'crow', 'raven', 'hawk', 'owl', 'scrub-jay']
PERSON_KEYWORDS = ['human', 'person', 'people']

for cid, name in ci_categories.items():
    name_lower = name.lower()
    if any(kw in name_lower for kw in RODENT_KEYWORDS):
        # Map to mouse or rat based on name
        CI_CLASS_MAP[cid] = 'rat' if 'rat' in name_lower else 'mouse'
    elif any(kw in name_lower for kw in CAT_KEYWORDS):
        CI_CLASS_MAP[cid] = 'cat'
    elif any(kw in name_lower for kw in DOG_KEYWORDS):
        CI_CLASS_MAP[cid] = 'dog'
    elif any(kw in name_lower for kw in BIRD_KEYWORDS):
        CI_CLASS_MAP[cid] = 'bird'
    elif any(kw in name_lower for kw in PERSON_KEYWORDS):
        CI_CLASS_MAP[cid] = 'person'

print('Category mappings:')
for cid, our_class in sorted(CI_CLASS_MAP.items()):
    print(f'  {ci_categories[cid]} -> {our_class}')

# Index: image_id -> annotations
ci_ann_by_image = defaultdict(list)
for ann in ci_coco['annotations']:
    if ann['category_id'] in CI_CLASS_MAP:
        ci_ann_by_image[ann['image_id']].append(ann)

# Filter to images that have mapped annotations
ci_images_with_ann = [img for img in ci_coco['images'] if img['id'] in ci_ann_by_image]
print(f'\nImages with relevant annotations: {len(ci_images_with_ann)}')

# Count per class
class_counts = defaultdict(int)
for img in ci_images_with_ann:
    for ann in ci_ann_by_image[img['id']]:
        class_counts[CI_CLASS_MAP[ann['category_id']]] += 1
print('\nAnnotation counts:')
for cls, cnt in sorted(class_counts.items()):
    print(f'  {cls}: {cnt}')

In [ ]:
# Cell 4: Download Channel Islands images (sample for training speed)
# We sample up to 5K rodent images and 2K of each other class

import urllib.request
from concurrent.futures import ThreadPoolExecutor, as_completed

CI_IMAGES_DIR = DATASET_DIR / 'images' / 'channel_islands'
CI_LABELS_DIR = DATASET_DIR / 'labels' / 'channel_islands'
CI_IMAGES_DIR.mkdir(parents=True, exist_ok=True)
CI_LABELS_DIR.mkdir(parents=True, exist_ok=True)

# CI images are on Azure blob storage
CI_IMAGE_BASE = 'https://lilablobssc.blob.core.windows.net/channel-islands-camera-traps/'

# Sample images per class
MAX_RODENT = 5000
MAX_OTHER = 2000

# Group images by primary class
images_by_class = defaultdict(list)
for img in ci_images_with_ann:
    # Use first annotation's class as primary
    primary = CI_CLASS_MAP[ci_ann_by_image[img['id']][0]['category_id']]
    images_by_class[primary].append(img)

selected_images = []
for cls, imgs in images_by_class.items():
    limit = MAX_RODENT if cls in ('mouse', 'rat') else MAX_OTHER
    sample = random.sample(imgs, min(limit, len(imgs)))
    selected_images.extend(sample)
    print(f'{cls}: selected {len(sample)} of {len(imgs)}')

random.shuffle(selected_images)
print(f'\nTotal selected: {len(selected_images)}')

def download_and_label(img_info):
    """Download image and create YOLO-format label file."""
    img_id = img_info['id']
    filename = img_info['file_name']
    w, h = img_info['width'], img_info['height']
    
    # Safe filename
    safe_name = filename.replace('/', '_').replace('\\', '_')
    img_path = CI_IMAGES_DIR / safe_name
    label_path = CI_LABELS_DIR / (Path(safe_name).stem + '.txt')
    
    if img_path.exists() and label_path.exists():
        return True
    
    # Download image
    try:
        url = CI_IMAGE_BASE + filename
        urllib.request.urlretrieve(url, img_path)
    except Exception:
        return False
    
    # Convert COCO bbox [x,y,w,h] to YOLO [cx,cy,w,h] normalized
    lines = []
    for ann in ci_ann_by_image[img_id]:
        cls_name = CI_CLASS_MAP.get(ann['category_id'])
        if not cls_name:
            continue
        cls_id = CLASS_TO_ID[cls_name]
        bx, by, bw, bh = ann['bbox']
        # COCO bbox is [x_min, y_min, width, height]
        cx = (bx + bw / 2) / w
        cy = (by + bh / 2) / h
        nw = bw / w
        nh = bh / h
        # Clamp
        cx = max(0, min(1, cx))
        cy = max(0, min(1, cy))
        nw = max(0, min(1, nw))
        nh = max(0, min(1, nh))
        lines.append(f'{cls_id} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}')
    
    label_path.write_text('\n'.join(lines))
    return True

# Download with thread pool
print('\nDownloading images...')
success = 0
with ThreadPoolExecutor(max_workers=16) as executor:
    futures = {executor.submit(download_and_label, img): img for img in selected_images}
    for i, future in enumerate(as_completed(futures)):
        if future.result():
            success += 1
        if (i + 1) % 500 == 0:
            print(f'  {i+1}/{len(selected_images)} ({success} ok)')

print(f'\nDownloaded {success} of {len(selected_images)} images')

In [ ]:
# Cell 5: Add COCO 2017 person/cat/dog/bird data
# Downloads COCO val2017 (5K images) and extracts relevant annotations

COCO_DIR = WORK_DIR / 'coco'
COCO_DIR.mkdir(exist_ok=True)

# COCO 2017 val annotations (smaller, faster to download)
COCO_ANN_URL = 'http://images.cocodataset.org/annotations/annotations_trainval2017.zip'
COCO_VAL_URL = 'http://images.cocodataset.org/zips/val2017.zip'

# Download annotations
ann_zip = COCO_DIR / 'annotations.zip'
if not ann_zip.exists():
    print('Downloading COCO annotations...')
    urllib.request.urlretrieve(COCO_ANN_URL, ann_zip)
    with zipfile.ZipFile(ann_zip) as zf:
        zf.extractall(COCO_DIR)

# Download val images
val_zip = COCO_DIR / 'val2017.zip'
if not val_zip.exists():
    print('Downloading COCO val2017 images...')
    urllib.request.urlretrieve(COCO_VAL_URL, val_zip)
    with zipfile.ZipFile(val_zip) as zf:
        zf.extractall(COCO_DIR)

# Load COCO val annotations
with open(COCO_DIR / 'annotations' / 'instances_val2017.json') as f:
    coco_val = json.load(f)

# COCO category IDs we care about
COCO_CLASS_MAP = {
    1: 'person',
    16: 'bird',
    17: 'cat',
    18: 'dog',
}

coco_images = {img['id']: img for img in coco_val['images']}

# Group annotations by image
coco_ann_by_image = defaultdict(list)
for ann in coco_val['annotations']:
    if ann['category_id'] in COCO_CLASS_MAP:
        coco_ann_by_image[ann['image_id']].append(ann)

# Convert COCO annotations to YOLO format
COCO_IMAGES_DIR = DATASET_DIR / 'images' / 'coco'
COCO_LABELS_DIR = DATASET_DIR / 'labels' / 'coco'
COCO_IMAGES_DIR.mkdir(parents=True, exist_ok=True)
COCO_LABELS_DIR.mkdir(parents=True, exist_ok=True)

coco_count = 0
for img_id, anns in coco_ann_by_image.items():
    if img_id not in coco_images:
        continue
    img_info = coco_images[img_id]
    w, h = img_info['width'], img_info['height']
    filename = img_info['file_name']
    
    src_path = COCO_DIR / 'val2017' / filename
    if not src_path.exists():
        continue
    
    # Copy image
    dst_path = COCO_IMAGES_DIR / filename
    if not dst_path.exists():
        shutil.copy2(src_path, dst_path)
    
    # Create YOLO label
    lines = []
    for ann in anns:
        cls_name = COCO_CLASS_MAP[ann['category_id']]
        cls_id = CLASS_TO_ID[cls_name]
        bx, by, bw, bh = ann['bbox']
        cx = (bx + bw / 2) / w
        cy = (by + bh / 2) / h
        nw = bw / w
        nh = bh / h
        cx = max(0, min(1, cx))
        cy = max(0, min(1, cy))
        nw = max(0, min(1, nw))
        nh = max(0, min(1, nh))
        lines.append(f'{cls_id} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}')
    
    label_path = COCO_LABELS_DIR / (Path(filename).stem + '.txt')
    label_path.write_text('\n'.join(lines))
    coco_count += 1

print(f'Processed {coco_count} COCO images with person/cat/dog/bird annotations')

In [ ]:
# Cell 6: Merge datasets and create train/val split
# Combines Channel Islands + COCO into a single YOLO dataset

FINAL_DIR = WORK_DIR / 'final_dataset'
for split in ['train', 'val']:
    (FINAL_DIR / 'images' / split).mkdir(parents=True, exist_ok=True)
    (FINAL_DIR / 'labels' / split).mkdir(parents=True, exist_ok=True)

# Collect all image-label pairs
all_pairs = []

for source in ['channel_islands', 'coco']:
    img_dir = DATASET_DIR / 'images' / source
    lbl_dir = DATASET_DIR / 'labels' / source
    if not img_dir.exists():
        continue
    for img_path in img_dir.iterdir():
        if img_path.suffix.lower() not in ('.jpg', '.jpeg', '.png'):
            continue
        lbl_path = lbl_dir / (img_path.stem + '.txt')
        if lbl_path.exists() and lbl_path.stat().st_size > 0:
            all_pairs.append((img_path, lbl_path, source))

random.shuffle(all_pairs)
print(f'Total image-label pairs: {len(all_pairs)}')

# 80/20 train/val split
split_idx = int(len(all_pairs) * 0.8)
train_pairs = all_pairs[:split_idx]
val_pairs = all_pairs[split_idx:]

print(f'Train: {len(train_pairs)}')
print(f'Val: {len(val_pairs)}')

def copy_pair(pair, split):
    img_path, lbl_path, source = pair
    # Prefix filename with source to avoid collisions
    new_name = f'{source}_{img_path.name}'
    shutil.copy2(img_path, FINAL_DIR / 'images' / split / new_name)
    shutil.copy2(lbl_path, FINAL_DIR / 'labels' / split / (f'{source}_{img_path.stem}.txt'))

for pair in train_pairs:
    copy_pair(pair, 'train')
for pair in val_pairs:
    copy_pair(pair, 'val')

print('\nDataset split complete.')

# Count labels per class in training set
train_class_counts = defaultdict(int)
for _, lbl_path, _ in train_pairs:
    for line in lbl_path.read_text().strip().split('\n'):
        if line.strip():
            cls_id = int(line.split()[0])
            train_class_counts[CLASS_NAMES[cls_id]] += 1

print('\nTrain label distribution:')
for cls in CLASS_NAMES:
    print(f'  {cls}: {train_class_counts.get(cls, 0)}')

In [ ]:
# Cell 7: Create YOLO dataset config and train

from ultralytics import YOLO
import yaml

# Write dataset.yaml for YOLO
dataset_config = {
    'path': str(FINAL_DIR),
    'train': 'images/train',
    'val': 'images/val',
    'names': {i: name for i, name in enumerate(CLASS_NAMES)},
    'nc': len(CLASS_NAMES),
}

config_path = FINAL_DIR / 'dataset.yaml'
with open(config_path, 'w') as f:
    yaml.dump(dataset_config, f, default_flow_style=False)

print('Dataset config:')
print(yaml.dump(dataset_config, default_flow_style=False))

# Load YOLOv8n pretrained on COCO (transfer learning)
model = YOLO('yolov8n.pt')

# Train
results = model.train(
    data=str(config_path),
    epochs=50,
    imgsz=640,
    batch=16,
    patience=10,        # Early stopping
    device=0,           # GPU
    workers=4,
    project=str(WORK_DIR / 'runs'),
    name='mousetrap_yolo',
    exist_ok=True,
    # Augmentation
    hsv_h=0.015,
    hsv_s=0.4,
    hsv_v=0.3,
    degrees=10,
    translate=0.1,
    scale=0.3,
    flipud=0.5,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    # Optimization
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
)

print('\nTraining complete!')

In [ ]:
# Cell 8: Evaluate the model

# Run validation
run_dir = WORK_DIR / 'runs' / 'mousetrap_yolo'
best_weights = run_dir / 'weights' / 'best.pt'

print(f'Best weights: {best_weights}')
print(f'File size: {best_weights.stat().st_size / 1024 / 1024:.1f} MB')

# Load best model and validate
best_model = YOLO(str(best_weights))
metrics = best_model.val(data=str(config_path), split='val')

print(f'\n=== Validation Results ===')
print(f'mAP50:     {metrics.box.map50:.4f}')
print(f'mAP50-95:  {metrics.box.map:.4f}')
print(f'\nPer-class AP50:')
for i, name in enumerate(CLASS_NAMES):
    if i < len(metrics.box.ap50):
        print(f'  {name:10s}: {metrics.box.ap50[i]:.4f}')

# Target: mAP50 > 0.85 for rodent classes
rodent_ids = [CLASS_TO_ID['mouse'], CLASS_TO_ID['rat']]
rodent_ap = [metrics.box.ap50[i] for i in rodent_ids if i < len(metrics.box.ap50)]
if rodent_ap:
    mean_rodent_ap = sum(rodent_ap) / len(rodent_ap)
    print(f'\nMean rodent AP50: {mean_rodent_ap:.4f}')
    if mean_rodent_ap > 0.85:
        print('✓ PASS: Rodent detection meets target (>0.85)')
    else:
        print('✗ Below target — consider more rodent training data or more epochs')

In [ ]:
# Cell 9: Confusion matrix and per-class precision/recall

import matplotlib.pyplot as plt
import numpy as np

# Plot confusion matrix if available
cm_path = run_dir / 'confusion_matrix_normalized.png'
if cm_path.exists():
    from IPython.display import Image, display
    print('Confusion Matrix (normalized):')
    display(Image(filename=str(cm_path), width=600))

# Plot training curves
results_csv = run_dir / 'results.csv'
if results_csv.exists():
    import pandas as pd
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # Loss
    if 'train/box_loss' in df.columns:
        axes[0].plot(df['train/box_loss'], label='train box')
        axes[0].plot(df['val/box_loss'], label='val box')
        axes[0].set_title('Box Loss')
        axes[0].legend()
    
    # mAP
    if 'metrics/mAP50(B)' in df.columns:
        axes[1].plot(df['metrics/mAP50(B)'], label='mAP50')
        axes[1].plot(df['metrics/mAP50-95(B)'], label='mAP50-95')
        axes[1].set_title('mAP')
        axes[1].legend()
    
    # Precision/Recall
    if 'metrics/precision(B)' in df.columns:
        axes[2].plot(df['metrics/precision(B)'], label='Precision')
        axes[2].plot(df['metrics/recall(B)'], label='Recall')
        axes[2].set_title('Precision / Recall')
        axes[2].legend()
    
    plt.tight_layout()
    plt.show()

print('\nPer-class Precision & Recall:')
for i, name in enumerate(CLASS_NAMES):
    if i < len(metrics.box.p) and i < len(metrics.box.r):
        print(f'  {name:10s}: P={metrics.box.p[i]:.3f}  R={metrics.box.r[i]:.3f}')

In [ ]:
# Cell 10: Export and download weights

from google.colab import files

# Copy best weights to a convenient name
export_path = WORK_DIR / 'best.pt'
shutil.copy2(best_weights, export_path)

print(f'Model exported to: {export_path}')
print(f'Size: {export_path.stat().st_size / 1024 / 1024:.1f} MB')
print(f'\nClasses: {CLASS_NAMES}')
print(f'mAP50: {metrics.box.map50:.4f}')

# Download
print('\n--- Download the trained model ---')
print('Place it at: classification-service/weights/best.pt')
files.download(str(export_path))

print('\nDone! Deploy with:')
print('  cp best.pt /path/to/MouseTrap/classification-service/weights/')
print('  cd /path/to/MouseTrap/classification-service && docker compose up --build -d')